# Likninger og nullpunkter

```{admonition} Læringsutbytte
Etter å ha arbeidet med dette temaet, skal du kunne:

1. formulere en likning som et nullpunktsproblem
2. forklare og implementere halveringsmetoden og Newtons metode
3. bruke en graf til å velge fornuftige startverdier og intervaller
4. vurdere konvergens, toleranse og begrensninger ved numeriske løsninger
5. bruke ferdige nullpunktsmetoder fra SciPy
6. bruke nullpunktsmetoder på kjemiske problemer
```

Mange kjemiske problemer kan formuleres som en likning. Noen kan løses analytisk, men i mer sammensatte modeller er det ofte enklere å finne løsningen numerisk.

En generell strategi er å skrive likningen på formen

$$f(x)=0.$$

Å løse likningen er da det samme som å finne et **nullpunkt** til funksjonen $f$.

## Fra kjemisk problem til nullpunkt

Som eksempel ser vi på en svak, enprotisk syre HA med startkonsentrasjon $c_0$. Dersom $h=[\mathrm{H_3O^+}]$, kan konsentrasjonen av den korresponderende basen skrives

$$[\mathrm{A^-}]=\frac{K_a c_0}{h+K_a}.$$

Sammen med vannets ionprodukt og ladningsbalansen får vi

$$h=[\mathrm{A^-}]+\frac{K_w}{h}.$$

Dermed kan pH-problemet skrives som nullpunktsproblemet

$$f(h)=h-\frac{K_a c_0}{h+K_a}-\frac{K_w}{h}=0.$$

Det er denne oversettelsen fra **kjemisk modell** til **matematisk likning** som er det viktigste steget.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

Ka = 1.75e-5
Kw = 1.0e-14
c0 = 0.010

def charge_balance(h):
    return h - Ka*c0/(h + Ka) - Kw/h

h_values = np.logspace(-5, -1, 400)

plt.semilogx(h_values, charge_balance(h_values))
plt.axhline(0)
plt.xlabel("[H3O+] (mol/L)")
plt.ylabel("f(h)")
plt.show()


Før vi starter en numerisk algoritme, er det ofte lurt å **plotte funksjonen**. Her ser vi omtrent hvor nullpunktet ligger. Grafen hjelper oss både med å forstå problemet og med å velge et fornuftig intervall eller startgjett.

## Halveringsmetoden

Halveringsmetoden bygger på et enkelt prinsipp: Dersom en kontinuerlig funksjon har ulikt fortegn i endepunktene $a$ og $b$, må den ha minst ett nullpunkt mellom dem.

Vi halverer intervallet og beholder den halvparten der fortegnet fortsatt skifter.

```{admonition} Halveringsmetoden
:class: note
Start med et intervall $[a,b]$ der $f(a)f(b)<0$.

1. Finn midtpunktet $c=(a+b)/2$.
2. Undersøk hvilken halvdel som inneholder et fortegnsskifte.
3. Gjenta til intervallet eller $|f(c)|$ er mindre enn ønsket toleranse.
```


In [ ]:
def bisection(f, a, b, tolerance=1e-10, max_iterations=100):
    if f(a) * f(b) > 0:
        raise ValueError("f(a) og f(b) må ha ulikt fortegn.")

    for _ in range(max_iterations):
        c = (a + b) / 2

        if abs(f(c)) < tolerance:
            return c

        if f(a) * f(c) < 0:
            b = c
        else:
            a = c

    raise RuntimeError("Metoden konvergerte ikke innen maks antall iterasjoner.")

h = bisection(charge_balance, 1e-5, 1e-1)
pH = -np.log10(h)

print(f"[H3O+] = {h:.4e} mol/L")
print(f"pH = {pH:.3f}")


**Underveisoppgave:** Endre startkonsentrasjonen `c0`. Hvordan påvirkes pH? Finn også ut hva som skjer dersom du velger et intervall som ikke omslutter nullpunktet.

Du kan prøve halveringsmetoden i editoren nedenfor.

<iframe src="../../basthon/?from=examples/numerical_bisection_acid.py" width="100%" height="600" frameborder="0" title="Basthon: halveringsmetoden" loading="lazy" allowfullscreen></iframe>

## Newtons metode

Halveringsmetoden er robust, men kan være langsom. Newtons metode bruker i stedet tangenten til funksjonen. Studentene kjenner notasjonen $f'(x)$ fra videregående. Den samme deriverte kan også skrives

$$f'(x)=\frac{df}{dx}.$$

Fra tangentlikningen får vi iterasjonen

$$x_{n+1}=x_n-\frac{f(x_n)}{f'(x_n)}.$$

Metoden trenger bare ett startgjett og konvergerer ofte svært raskt. Til gjengjeld kan et dårlig startgjett gi problemer, og metoden fungerer ikke dersom den deriverte blir null underveis.


In [ ]:
def newton(f, df, x0, tolerance=1e-10, max_iterations=50):
    x = x0

    for _ in range(max_iterations):
        slope = df(x)

        if abs(slope) < 1e-14:
            raise RuntimeError("Den deriverte er for nær null.")

        x_new = x - f(x) / slope

        if abs(x_new - x) < tolerance:
            return x_new

        x = x_new

    raise RuntimeError("Metoden konvergerte ikke innen maks antall iterasjoner.")

def f(x):
    return x**3 - 2*x - 5

def df(x):
    return 3*x**2 - 2

root_newton = newton(f, df, 2.0)
print(root_newton)


### Når kan Newtons metode mislykkes?

Tre vanlige problemer er:

- startgjettet ligger ugunstig til
- $f'(x)$ blir svært liten eller lik null
- funksjonen har flere nullpunkter, slik at ulike startgjett kan gi ulike løsninger

Dette er en viktig påminnelse: Et tall fra en algoritme er ikke automatisk et godt svar. Vi bør kontrollere løsningen ved å sette den inn i funksjonen og vurdere om den er kjemisk rimelig.

## Ferdige metoder i SciPy

Når vi har forstått prinsippene, er det vanlig å bruke ferdige og grundig testede algoritmer. `root_scalar` samler flere nullpunktsmetoder.


In [ ]:
from scipy.optimize import root_scalar

solution = root_scalar(
    charge_balance,
    bracket=[1e-5, 1e-1],
    method="bisect"
)

h = solution.root
print(f"pH = {-np.log10(h):.3f}")
print("Konvergert:", solution.converged)


## Numerisk svar er ikke nok

Når vi løser et kjemisk problem numerisk, bør vi alltid spørre:

1. **Modell:** Er likningen en rimelig beskrivelse av kjemien?
2. **Algoritme:** Passer metoden til problemet?
3. **Numerikk:** Har løsningen konvergert, og er toleransen fornuftig?
4. **Kontroll:** Oppfyller løsningen likningen?
5. **Kjemi:** Er svaret fysisk og kjemisk mulig?

## Oppgaver

```{admonition} Oppgave 1 – fra likning til nullpunkt
:class: tip
Skriv likningen $e^{-x}+x=2$ som et nullpunktsproblem. Plott funksjonen og finn et intervall som omslutter et nullpunkt.
```

```{admonition} Oppgave 2 – halveringsmetoden
:class: tip
Bruk din egen halveringsmetode til å løse $x^5=5x^3+3$. Finn alle de tre reelle løsningene ved å velge tre ulike intervaller.
```

```{admonition} Oppgave 3 – Newton og startgjett
:class: tip
Bruk Newtons metode på $f(x)=x^3-2x+2$ med flere ulike startgjett. Undersøk hva som skjer. Hvorfor er Newtons metode mindre robust enn halveringsmetoden i dette eksemplet?
```

```{admonition} Oppgave 4 – svak syre
:class: tip
Bruk nullpunktsmodellen i kapitlet til å beregne pH i 0,0250 M eddiksyre med $K_a=1,75\cdot10^{-5}$. Sammenlikn din egen halveringsmetode med `root_scalar`.
```

```{admonition} Oppgave 5 – likevekt
:class: tip
For reaksjonen $\mathrm{A \rightleftharpoons B}$ er startkonsentrasjonene $[A]_0=0,80$ M og $[B]_0=0,10$ M. Ved likevekt er $K_c=3,5$. La reaksjonsframgangen være $x$, slik at $[A]=0,80-x$ og $[B]=0,10+x$. Formuler $K_c=[B]/[A]$ som et nullpunktsproblem og finn $x$ numerisk. Kontroller at alle konsentrasjonene blir positive.
```

```{admonition} Oppgave 6 – velg metode
:class: tip
Du har to nullpunktsproblemer: ett der du kjenner et sikkert fortegnsskifte, og ett der funksjonen er svært dyr å evaluere, men du har et godt startgjett og kjenner den deriverte. Hvilken metode ville du valgt i hvert tilfelle? Begrunn.
```
